<center><h1>🧗‍♂️ Bouldering Comp Simulation</h1></center>

The idea for this small prototype is inspired by climbing competitions (see [IFSC](https://en.wikipedia.org/wiki/International_Federation_of_Sport_Climbing)).\
Given a list of competitors and a boulder problem, the system will generate (hopefully mostly funny) commentary, describing how each climber is performing on the current boulder problem. The comments should be based on the climber's _style_ and _level_ of climbing - the less advanced a climber is, the funnier their commentary is likely to be.\
You can try this with friends - insert your specific information (height, level of climbing - could be also `0`, if you have never climbed) and see what commentary is generated for your simulated comp. 🙂 

--- 


### How to run this notebook

- Create virtual env `my_env` and run it: `source my_env/bin/activate` (optional).
- Install the dependencies in your active env or just directly. You can do it in your terminal or in a new cell in this jupyter notebook like so: `!pip install my_dependency`.
- List of dependecies:
    - `sklearn`
    - `pandas`
    - `numpy`
    - `matplotlib`
    - `openai`
    - `dotenv`
- Add an `.env` file in your repository and add there your OpenAI key with `OPENAI_API_KEY="your_key_here"`.
- Activate the jupyter notebook: `jupyter notebook`.
- Run the notebook, I prefer to do it with `Kernel > Restart Kernel and Run All Cells`.
- Open the link for your localhost to see and interact with the prototype in the browser.


In [1]:
import gradio as gr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import openai
import os
import io
from PIL import Image
from dotenv import load_dotenv

In [2]:
# Get the OpenAI API key from your .env
load_dotenv()

openai_key = os.getenv("OPENAI_API_KEY")

In [3]:
# Setup OpenAI

client = openai.OpenAI(api_key=openai_key)

## Data

In [4]:
# Example climbers.
# These examples were generated by ChatGPT, replace with your own if you wish.

# This is just a prototype, for further development it would be to nice to be able to add climbers via the UI.

climbers = [
    {"name": "Lena Grip", "height_cm": 164, "specialty": "slab", "level": 5, "humor": "sassy"},
    {"name": "Tom Crux", "height_cm": 178, "specialty": "overhang", "level": 7, "humor": "serious"},
    {"name": "Anna Dyno", "height_cm": 160, "specialty": "dynamic", "level": 6, "humor": "goofy"},
    {"name": "Max Jugson", "height_cm": 185, "specialty": "power", "level": 8, "humor": "dry"},
    {"name": "Sasha Smear", "height_cm": 170, "specialty": "technical", "level": 4, "humor": "sarcastic"},
    {"name": "Bo Slipper", "height_cm": 175, "specialty": "slab", "level": 3, "humor": "naive"},
    {"name": "Karo Pinch", "height_cm": 168, "specialty": "pinches", "level": 6, "humor": "sharp"},
    {"name": "Jim Mantle", "height_cm": 180, "specialty": "mantle", "level": 2, "humor": "deadpan"},
]

In [5]:
# Use this to test the proptotype if you do not have an idea about a boulder problem.
# You can just copy and past the information from here in the UI.

boulder_problem_test = {
    "name": "The Banana Traverse",
    "description": "A long, slopey traverse with a dynamic finish and a tricky toe hook start.",
    "style": "slab/dyno combo",
    "grade": "V6"
}

## Functionalities

In [6]:
def generate_commentary(climber, boulder):
    prompt = f"""
        You're a witty IFSC climbing commentator.
        Climber info:
        - Name: {climber['name']}
        - Height: {climber['height_cm']} cm
        - Specialty: {climber['specialty']}
        - Level: {climber['level']}
        - Humor style: {climber['humor']}
        
        Boulder problem:
        - Name: {boulder['name']}
        - Description: {boulder['description']}
        - Style: {boulder['style']}
        - Grade: {boulder['grade']}
        
        Write a short, funny 3-sentence commentary on {climber['name']}'s attempt. If their level is low, lean into the humor.
    """

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a humorous sports commentator."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.9,
        max_tokens=150,
    )

    return response.choices[0].message.content

In [7]:
# Use the provided functions from sklearn

def plot_climber_clusters(df):
    X = StandardScaler().fit_transform(df[['height_cm', 'level']])
    kmeans = KMeans(n_clusters=3, random_state=42).fit(X)
    df['cluster'] = kmeans.labels_

    plt.figure(figsize=(6, 4))
    plt.scatter(X[:, 0], X[:, 1], c=kmeans.labels_, cmap='tab10')
    for i, name in enumerate(df['name']):
        plt.annotate(name, (X[i, 0], X[i, 1]))

    # Add plot info
    plt.title("Climbers clustered by height & level")
    plt.xlabel("Height (std)")
    plt.ylabel("Level (std)")

    plt.tight_layout()

    buf = io.BytesIO()
    plt.savefig(buf, format='png')
    plt.close()
    buf.seek(0)

    # Return the plot as an image
    return Image.open(buf)

In [8]:
def simulate_climb(name, description, style, grade):
    boulder = {
        "name": name,
        "description": description,
        "style": style,
        "grade": grade
    }

    df = pd.DataFrame(climbers)
    scores = []
    commentary = []

    for _, row in df.iterrows():
        climber = row.to_dict()
        score = round(min(max(climber['level'] + np.random.normal(0, 1.5), 0), 10), 2)
        comment = generate_commentary(climber, boulder)
        scores.append(score)
        commentary.append(f"🧗 {climber['name']} (Score: {score}):\n{comment.strip()}")

    df['score'] = scores
    df['commentary'] = commentary
    winner = df.sort_values(by='score', ascending=False).iloc[0]['name']
    cluster_img = plot_climber_clusters(df)

    result_text = "\n\n".join(commentary)
    result_text += f"\n\n🏆 **Winner:** {winner}"

    return result_text, cluster_img

## Create the UI

In [9]:
# The gradio UI 

gr.Interface(
    fn=simulate_climb,
    inputs=[
        gr.Textbox(label="Boulder name"),
        gr.Textbox(lines=3, label="Boulder Description"),
        gr.Textbox(label="Boulder Style (e.g. slab, dyno, compression)"),
        gr.Textbox(label="Boulder Grade (e.g. V5, V7, 6C+)"),
    ],
    outputs=[
        gr.Textbox(label="Commentary & Results"),
        gr.Image(type="pil", label="Clustering Visualization"),
    ],
    title="🧗‍♀️ Bouldering competition simulation",
    description="Simulate a bouldering comp with ML stats and ChatGPT-generated commentary."
).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
